# FABRIC Resource Finder

Query FABRIC testbed sites for available resources.

## Features:
- Find sites with specific NICs
- Find sites with GPUs, FPGAs, NVMe
- Filter by multiple criteria
- Export results to various formats

## 1. Setup

In [1]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

try:
    fablib = fablib_manager()
    print("✅ Fablib initialized successfully")
    fablib.show_config()
except Exception as e:
    print(f"❌ Exception: {e}")
    

User: mcevik@email.unc.edu bastion key is valid!
Configuration is valid
✅ Fablib initialized successfully


Credential Manager,cm.fabric-testbed.net
Orchestrator,orchestrator.fabric-testbed.net
Project ID,6b76128d-c73f-431f-a245-0397586a7d40
Token File,/home/fabric/.tokens.json
Bastion Host,bastion-ncsa-1.fabric-testbed.net
Bastion Username,mcevik_0000100641
Bastion Private Key File,/home/fabric/work/fabric_config/mcevik_0000100641_bastion
Slice Private Key File,/home/fabric/work/fabric_config/mcevik_0000100641_sliver
Slice Public Key File,/home/fabric/work/fabric_config/mcevik_0000100641_sliver.pub
Log File,/tmp/fablib/fablib.log
Log Level,INFO


## 2. Helper Function - Get Sites DataFrame

In [2]:
import pandas as pd

def get_sites_dataframe(fablib, force_refresh=False):
    """
    Get all FABRIC sites with their resources as a DataFrame.
    
    Args:
        fablib: FablibManager instance
        
    Returns:
        pandas.DataFrame with site information and resources
    """
    resources = fablib.get_resources(force_refresh=force_refresh)
    sites_list = []

    for site_name in resources.sites:
        site = resources.sites[site_name]
        
        # Build site dictionary
        site_dict = {'name': site_name}
        
        # Basic attributes
        try:
            site_dict['state'] = site.get_state()
        except:
            site_dict['state'] = None
        
        # Get location info
        try:
            location = site.get_location_postal()
            site_dict['address'] = location
        except:
            site_dict['address'] = None
        
        # Get basic resources
        try:
            site_dict['cores_available'] = site.get_core_available()
            site_dict['cores_capacity'] = site.get_core_capacity()
            site_dict['cores_allocated'] = site.get_core_allocated()
            
            site_dict['ram_available'] = site.get_ram_available()
            site_dict['ram_capacity'] = site.get_ram_capacity()
            site_dict['ram_allocated'] = site.get_ram_allocated()
            
            site_dict['disk_available'] = site.get_disk_available()
            site_dict['disk_capacity'] = site.get_disk_capacity()
            site_dict['disk_allocated'] = site.get_disk_allocated()
        except:
            pass
        
        # Get component info from site_info (NICs, GPUs, FPGAs, NVMe, etc.)
        try:
            site_info = site.site_info
            for component_name, component_data in site_info.items():
                if isinstance(component_data, dict) and 'capacity' in component_data:
                    # Create columns for each component
                    site_dict[f'{component_name}_capacity'] = component_data.get('capacity', 0)
                    site_dict[f'{component_name}_allocated'] = component_data.get('allocated', 0)
                    site_dict[f'{component_name}_available'] = (
                        component_data.get('capacity', 0) - component_data.get('allocated', 0)
                    )
        except Exception as e:
            pass
        
        sites_list.append(site_dict)

    return pd.DataFrame(sites_list)


## 3. Query - Sites with SmartNIC ConnectX-5

In [3]:
print("🔎 Sites with SmartNIC ConnectX-5 available:\n")

sites_df = get_sites_dataframe(fablib)
cx5_sites = sites_df[sites_df['smartnic-connectx-5_available'].fillna(0) > 0]

display_cols = ['name', 'smartnic-connectx-5_available']
print(cx5_sites[display_cols].to_string(index=False))


🔎 Sites with SmartNIC ConnectX-5 available:

  name  smartnic-connectx-5_available
  PRIN                            2.0
  ATLA                            3.0
  STAR                            4.0
  SEAT                            3.0
  SALT                            1.0
   GPN                            3.0
   FIU                            1.0
  TACC                            4.0
  CERN                            4.0
  CLEM                            2.0
   MAX                            4.0
GATECH                            4.0
  LOSA                            2.0
  RUTG                            4.0
  NCSA                            1.0
   PSC                            2.0
  UTAH                            4.0
  NEWY                            3.0
  MICH                            1.0
  UCSD                            2.0
  MASS                            1.0
  WASH                            2.0
  KANS                            2.0
  AMST                            2.0
 BRIS

## 4. Query - Sites with SmartNIC ConnectX-6

In [4]:
print("🔎 Sites with SmartNIC ConnectX-6 available:\n")

sites_df = get_sites_dataframe(fablib)
cx6_sites = sites_df[sites_df['smartnic-connectx-6_available'].fillna(0) > 0]

display_cols = ['name', 'smartnic-connectx-6_available']
print(cx6_sites[display_cols].to_string(index=False))


🔎 Sites with SmartNIC ConnectX-6 available:

  name  smartnic-connectx-6_available
  PRIN                            2.0
  ATLA                            2.0
  STAR                            2.0
  SEAT                            3.0
  SALT                            1.0
   GPN                            2.0
   FIU                            2.0
  TACC                            1.0
   MAX                            2.0
GATECH                            2.0
  LOSA                            1.0
  RUTG                            2.0
  UTAH                            2.0
  NEWY                            2.0
  MICH                            1.0
  UCSD                            1.0
  MASS                            2.0
  WASH                            2.0
  KANS                            1.0
  AMST                            2.0
 BRIST                            3.0
   SRI                            2.0
  HAWI                            1.0
  INDI                            1.0
  DAL

## 5. Query - Sites with SharedNIC ConnectX-6

In [5]:
print("🔎 Sites with SharedNIC ConnectX-6 available:\n")

sites_df = get_sites_dataframe(fablib)
shared_cx6_sites = sites_df[sites_df['sharednic-connectx-6_available'].fillna(0) > 0]

display_cols = ['name', 'sharednic-connectx-6_available']
print(shared_cx6_sites[display_cols].to_string(index=False))


🔎 Sites with SharedNIC ConnectX-6 available:

  name  sharednic-connectx-6_available
 EDUKY                          4429.0
  PRIN                           374.0
  ATLA                           246.0
  STAR                           672.0
  SEAT                           216.0
  SALT                           355.0
   GPN                           126.0
   FIU                           594.0
  TACC                           595.0
  CERN                           742.0
  CLEM                           340.0
   MAX                           555.0
GATECH                           572.0
  LOSA                           339.0
  RUTG                           591.0
  NCSA                           277.0
   PSC                           346.0
  UTAH                           373.0
  NEWY                           233.0
  MICH                           350.0
  UCSD                           575.0
  MASS                           344.0
  WASH                           358.0
  KANS            

## 6. Query - Sites with DPU NICs (ConnectX-7 100G, ConnectX-7 400G)

In [6]:
print("🔎 Sites with DPU NICs (ConnectX-7 100G) available:\n")

sites_df = get_sites_dataframe(fablib)
dpu_sites = sites_df[sites_df['smartnic-connectx-7-100_available'].fillna(0) > 0]

display_cols = ['name', 'smartnic-connectx-7-100_available']
print(dpu_sites[display_cols].to_string(index=False))


🔎 Sites with DPU NICs (ConnectX-7 100G) available:

name  smartnic-connectx-7-100_available
SEAT                                1.0
 FIU                                1.0
TACC                                1.0
LOSA                                1.0
NCSA                                1.0


In [7]:
print("🔎 Sites with DPU NICs (ConnectX-7 400G) available:\n")

sites_df = get_sites_dataframe(fablib)
dpu_sites = sites_df[sites_df['smartnic-connectx-7-400_available'].fillna(0) > 0]

display_cols = ['name', 'smartnic-connectx-7-400_available']
print(dpu_sites[display_cols].to_string(index=False))


🔎 Sites with DPU NICs (ConnectX-7 400G) available:

name  smartnic-connectx-7-400_available
SALT                                1.0
WASH                                1.0


## 7. Query - Sites with BOTH SmartNIC ConnectX-5 and ConnectX-6

In [8]:
print("🔎 Sites with both SmartNIC ConnectX-5 and ConnectX-6:\n")

sites_df = get_sites_dataframe(fablib)
both_nics_sites = sites_df[
    (sites_df['smartnic-connectx-5_available'].fillna(0) > 0) & 
    (sites_df['smartnic-connectx-6_available'].fillna(0) > 0)
]

display_cols = ['name', 'smartnic-connectx-5_available', 'smartnic-connectx-6_available']
print(both_nics_sites[display_cols].to_string(index=False))


🔎 Sites with both SmartNIC ConnectX-5 and ConnectX-6:

  name  smartnic-connectx-5_available  smartnic-connectx-6_available
  PRIN                            2.0                            2.0
  ATLA                            3.0                            2.0
  STAR                            4.0                            2.0
  SEAT                            3.0                            3.0
  SALT                            1.0                            1.0
   GPN                            3.0                            2.0
   FIU                            1.0                            2.0
  TACC                            4.0                            1.0
   MAX                            4.0                            2.0
GATECH                            4.0                            2.0
  LOSA                            2.0                            1.0
  RUTG                            4.0                            2.0
  UTAH                            4.0           

## 8. Query - Sites with GPUs

In [9]:
print("🔎 Sites with RTX6000 GPUs:\n")

sites_df = get_sites_dataframe(fablib)
rtx6000_sites = sites_df[sites_df['gpu-rtx6000_available'].fillna(0) > 0]

display_cols = ['name', 'gpu-rtx6000_available']
print(rtx6000_sites[display_cols].to_string(index=False))

print("\n🔎 Sites with Tesla T4 GPUs:\n")

t4_sites = sites_df[sites_df['gpu-tesla t4_available'].fillna(0) > 0]
display_cols = ['name', 'gpu-tesla t4_available']
print(t4_sites[display_cols].to_string(index=False))

print("\n🔎 Sites with A30 GPUs:\n")

a30_sites = sites_df[sites_df['gpu-a30_available'].fillna(0) > 0]
display_cols = ['name', 'gpu-a30_available']
print(a30_sites[display_cols].to_string(index=False))


🔎 Sites with RTX6000 GPUs:

name  gpu-rtx6000_available
STAR                    3.0
SALT                    2.0
 GPN                    3.0
 FIU                    3.0
TACC                    3.0
CLEM                    1.0
NCSA                    1.0
UTAH                    3.0
DALL                    1.0

🔎 Sites with Tesla T4 GPUs:

name  gpu-tesla t4_available
STAR                     1.0
 GPN                     1.0
 FIU                     1.0
TACC                     2.0
UTAH                     3.0
MICH                     2.0
UCSD                     2.0
WASH                     2.0
DALL                     2.0

🔎 Sites with A30 GPUs:

  name  gpu-a30_available
GATECH                2.0
  LOSA                4.0
  RUTG                5.0
  KANS                1.0
  AMST                3.0
 BRIST                1.0
   SRI                4.0
  HAWI                2.0
  INDI                1.0
  TOKY                4.0


## 9. Query - Sites with FPGAs

In [10]:
print("🔎 Sites with Xilinx U280 FPGAs:\n")

sites_df = get_sites_dataframe(fablib)
fpga_sites = sites_df[sites_df['fpga-xilinx-u280_available'].fillna(0) > 0]

display_cols = ['name', 'fpga-xilinx-u280_available']
print(fpga_sites[display_cols].to_string(index=False))


🔎 Sites with Xilinx U280 FPGAs:

  name  fpga-xilinx-u280_available
  PRIN                         1.0
   GPN                         1.0
   FIU                         1.0
  TACC                         1.0
  CLEM                         1.0
   MAX                         1.0
GATECH                         1.0
  RUTG                         1.0
  NCSA                         1.0
   PSC                         1.0
  UTAH                         1.0
  MICH                         1.0
  UCSD                         1.0
  MASS                         1.0
  KANS                         1.0
   SRI                         1.0
  HAWI                         1.0
  INDI                         1.0


## 10. Query - Sites with NVMe Storage

In [11]:
print("🔎 Sites with NVMe storage:\n")

sites_df = get_sites_dataframe(fablib)
nvme_sites = sites_df[sites_df['nvme-p4510_available'].fillna(0) > 0]

display_cols = ['name', 'nvme-p4510_available']
print(nvme_sites[display_cols].to_string(index=False))


🔎 Sites with NVMe storage:

  name  nvme-p4510_available
  PRIN                  10.0
  ATLA                   8.0
  STAR                  15.0
  SEAT                   8.0
  SALT                  10.0
   GPN                  16.0
   FIU                  16.0
  TACC                  16.0
  CERN                  20.0
  CLEM                  10.0
   MAX                  16.0
GATECH                  10.0
  LOSA                  10.0
  RUTG                  14.0
  NCSA                  10.0
   PSC                  10.0
  UTAH                  15.0
  NEWY                   8.0
  MICH                  10.0
  UCSD                  14.0
  MASS                  10.0
  WASH                   9.0
  KANS                   8.0
  AMST                  10.0
 BRIST                   8.0
   SRI                  10.0
  HAWI                  16.0
  INDI                  10.0
  DALL                  10.0
  TOKY                  10.0


## 11. Custom Resource Finder Function

### Resources of the Sites

In [12]:
# ── Shared resource field map ─────────────────────────────────────────────────
# Single source of truth for every resource type understood by the finder
# functions.  Each row is:
#   (param_name, display_label, site_field, host_field)
#
# site_field  : column name produced by get_sites_dataframe()
# host_field  : field name used by fablib.list_hosts()
#
# NOTE: GPU names differ between the two levels — the map makes that explicit.
# ─────────────────────────────────────────────────────────────────────────────

_RESOURCE_FIELD_MAP = [
    # param_name               label                  site_field                           host_field
    ('min_cores',               'Cores',               'cores_available',                   'cores_available'),
    ('min_ram',                 'RAM (GB)',             'ram_available',                     'ram_available'),
    ('min_disk',                'Disk (GB)',            'disk_available',                    'disk_available'),
    ('smartnic_connectx_5',     'SmartNIC ConnectX-5', 'smartnic-connectx-5_available',     'smartnic-connectx-5_available'),
    ('smartnic_connectx_6',     'SmartNIC ConnectX-6', 'smartnic-connectx-6_available',     'smartnic-connectx-6_available'),
    ('smartnic_connectx_7_100', 'SmartNIC CX7-100G',   'smartnic-connectx-7-100_available', 'smartnic-connectx-7-100_available'),
    ('smartnic_connectx_7_400', 'SmartNIC CX7-400G',   'smartnic-connectx-7-400_available', 'smartnic-connectx-7-400_available'),
    ('sharednic_connectx_6',    'SharedNIC ConnectX-6','sharednic-connectx-6_available',    'sharednic-connectx-6_available'),
    ('gpu_rtx6000',             'RTX6000',             'gpu-rtx6000_available',             'rtx6000_available'),
    ('gpu_a30',                 'A30',                 'gpu-a30_available',                 'a30_available'),
    ('gpu_a40',                 'A40',                 'gpu-a40_available',                 'a40_available'),
    ('gpu_tesla_t4',            'Tesla T4',            'gpu-tesla t4_available',            'tesla_t4_available'),
    ('fpga_u280',               'U280 FPGA',           'fpga-xilinx-u280_available',        'fpga-xilinx-u280_available'),
    ('nvme',                    'NVMe',                'nvme-p4510_available',              'nvme-p4510_available'),
]

# Quick lookup: param_name -> (label, site_field, host_field)
_FIELD_LOOKUP = {p: (lbl, sf, hf) for p, lbl, sf, hf in _RESOURCE_FIELD_MAP}


def _print_criteria(level, criteria):
    """Print a formatted search-criteria header (shared by both finder functions)."""
    print(f"\U0001f50e Searching for {level} with:")
    for param, (label, *_) in _FIELD_LOOKUP.items():
        val = criteria.get(param, 0)
        if val:
            print(f"   \u2022 {label}: >= {val}")
    print()


In [13]:
def find_sites_with_resources(
    fablib,
    min_cores=0,
    min_ram=0,
    min_disk=0,
    smartnic_connectx_5=0,
    smartnic_connectx_6=0,
    smartnic_connectx_7_100=0,
    smartnic_connectx_7_400=0,
    sharednic_connectx_6=0,
    gpu_rtx6000=0,
    gpu_a30=0,
    gpu_a40=0,
    gpu_tesla_t4=0,
    fpga_u280=0,
    nvme=0,
    verbose=True,
    force_refresh=False,
    return_data=False
):
    """
    Find FABRIC sites matching resource criteria.

    Each returned row represents one site whose *aggregate* available
    resources satisfy all specified thresholds.

    Args:
        fablib: FablibManager instance
        min_cores / min_ram / min_disk: minimum compute resources
        smartnic_connectx_5/6/7_100/7_400: minimum SmartNIC counts
        sharednic_connectx_6: minimum SharedNIC count
        gpu_rtx6000 / gpu_a30 / gpu_a40 / gpu_tesla_t4: minimum GPU counts
        fpga_u280: minimum FPGA count
        nvme: minimum NVMe device count
        verbose: print search criteria header
        force_refresh: bypass the cache and fetch live data from the testbed
        return_data: if True, return the filtered DataFrame for programmatic use

    Returns:
        pandas.DataFrame if return_data=True, else None
    """
    # Collect criteria into a dict keyed by param name
    criteria = {p: v for p, v in locals().items() if p in _FIELD_LOOKUP}

    if verbose:
        _print_criteria('sites', criteria)

    sites_df = get_sites_dataframe(fablib, force_refresh=force_refresh)

    def safe_get(row, col, default=0):
        val = row.get(col, default)
        return default if pd.isna(val) else val

    def filter_func(row):
        return all(
            safe_get(row, site_field) >= criteria[param]
            for param, (_, site_field, _hf) in _FIELD_LOOKUP.items()
        )

    result_df = sites_df[sites_df.apply(filter_func, axis=1)]

    # Build display columns: always show name + state, then requested resources
    display_cols = ['name', 'state'] + [
        site_field
        for param, (_, site_field, _hf) in _FIELD_LOOKUP.items()
        if criteria[param] and site_field in result_df.columns
    ]

    print(f"\u2705 Found {len(result_df)} matching site(s):\n")
    if len(result_df) > 0:
        print(result_df[display_cols].to_string(index=False))
    else:
        print("No sites match the specified criteria.")

    if return_data:
        return result_df


#### Example: Find sites with at least 8 cores, 32GB RAM, and SharedNIC ConnectX-6

In [14]:
find_sites_with_resources(
    fablib,
    min_cores=256,
    min_ram=300,
    min_disk=500,
    sharednic_connectx_6=1,
    force_refresh=True,
    return_data=False
)

🔎 Searching for sites with:
   • Cores: >= 256
   • RAM (GB): >= 300
   • Disk (GB): >= 500
   • SharedNIC ConnectX-6: >= 1

✅ Found 6 matching site(s):

 name  state  cores_available  ram_available  disk_available  sharednic-connectx-6_available
EDUKY Active            73704           8416          132972                          4429.0
 PRIN Active              346           1250            3848                           374.0
 STAR Active              282           1396           95146                           672.0
 CERN Active              330           1316           43183                           742.0
 RUTG Active              518           1942          103853                           591.0
 HAWI Active              376           1358          102033                           610.0


In [15]:
find_sites_with_resources(fablib, min_cores=100, min_ram=200, gpu_a30=1, force_refresh=True, return_data=False)


🔎 Searching for sites with:
   • Cores: >= 100
   • RAM (GB): >= 200
   • A30: >= 1

✅ Found 10 matching site(s):

  name  state  cores_available  ram_available  gpu-a30_available
GATECH Active              244           1562                2.0
  LOSA Active              124            770                4.0
  RUTG Active              518           1942                5.0
  KANS Active              146            786                1.0
  AMST Active              202            946                3.0
 BRIST Active              236            970                1.0
   SRI Active              220            830                4.0
  HAWI Active              376           1358                2.0
  INDI Active              136            866                1.0
  TOKY Active              172            892                4.0


### Resources of the Hosts (Workers)

In [16]:
def find_hosts_with_resources(
    fablib,
    min_cores=0,
    min_ram=0,
    min_disk=0,
    smartnic_connectx_5=0,
    smartnic_connectx_6=0,
    smartnic_connectx_7_100=0,
    smartnic_connectx_7_400=0,
    sharednic_connectx_6=0,
    gpu_rtx6000=0,
    gpu_a30=0,
    gpu_a40=0,
    gpu_tesla_t4=0,
    fpga_u280=0,
    nvme=0,
    verbose=True,
    force_refresh=False,
    return_data=False
):
    """
    Find individual FABRIC worker hosts matching resource criteria.

    Each result is a single physical host that satisfies ALL specified
    thresholds on its own (not aggregate across the site).

    Note: GPU field names differ between site-level and host-level APIs;
          _RESOURCE_FIELD_MAP captures both so the caller never needs to
          worry about the difference.

    Args:
        fablib: FablibManager instance
        min_cores / min_ram / min_disk: minimum compute resources
        smartnic_connectx_5/6/7_100/7_400: minimum SmartNIC counts
        sharednic_connectx_6: minimum SharedNIC count
        gpu_rtx6000 / gpu_a30 / gpu_a40 / gpu_tesla_t4: minimum GPU counts
        fpga_u280: minimum FPGA count
        nvme: minimum NVMe device count
        verbose: print search criteria header
        force_refresh: bypass the cache and fetch live data from the testbed
        return_data: if True, return the list of host dicts for programmatic use

    Returns:
        list of dicts if return_data=True, else None
    """
    criteria = {p: v for p, v in locals().items() if p in _FIELD_LOOKUP}

    if verbose:
        _print_criteria('hosts', criteria)

    # Always fetch name + state + compute; add requested resource fields
    host_fields = ['name', 'state', 'cores_available', 'ram_available', 'disk_available'] + [
        host_field
        for param, (_, _sf, host_field) in _FIELD_LOOKUP.items()
        if criteria[param] and host_field not in ('cores_available', 'ram_available', 'disk_available')
    ]

    def host_filter(row):
        return all(
            row.get(host_field, 0) >= criteria[param]
            for param, (_, _sf, host_field) in _FIELD_LOOKUP.items()
        )

    try:
        matching_hosts = fablib.list_hosts(
            fields=host_fields,
            pretty_names=False,
            filter_function=host_filter,
            output='list',
            quiet=True,
            force_refresh=force_refresh
        )
    except Exception as e:
        print(f"\u274c Error querying hosts: {e}")
        return []

    print(f"\u2705 Found {len(matching_hosts)} matching host(s):\n")
    if matching_hosts:
        for host in matching_hosts:
            parts = [f"  {host.get('name', 'N/A')}  |"]
            parts.append(f"cores: {host.get('cores_available', 'N/A')}")
            parts.append(f"ram: {host.get('ram_available', 'N/A')} GB")
            parts.append(f"disk: {host.get('disk_available', 'N/A')} GB")
            for param, (label, _sf, host_field) in _FIELD_LOOKUP.items():
                if criteria[param] and host_field not in ('cores_available', 'ram_available', 'disk_available'):
                    parts.append(f"{label}: {host.get(host_field, 0)}")
            print("  ".join(parts))
    else:
        print("  No individual hosts match the specified criteria.")

    if return_data:
        return matching_hosts


#### Example: Find hosts with at least 8 cores, 32GB RAM, and an A30 GPU

In [17]:
find_hosts_with_resources(
    fablib,
    min_cores=8,
    min_ram=32,
    gpu_a30=1,
    force_refresh=True,
    return_data=False
)

🔎 Searching for hosts with:
   • Cores: >= 8
   • RAM (GB): >= 32
   • A30: >= 1

✅ Found 17 matching host(s):

  gatech-w1.fabric-testbed.net  |  cores: 16  ram: 286 GB  disk: 25375 GB  A30: 1
  gatech-w4.fabric-testbed.net  |  cores: 90  ram: 410 GB  disk: 1463 GB  A30: 1
  losa-w3.fabric-testbed.net  |  cores: 40  ram: 286 GB  disk: 433 GB  A30: 1
  losa-w1.fabric-testbed.net  |  cores: 52  ram: 342 GB  disk: 49052 GB  A30: 3
  rutg-w1.fabric-testbed.net  |  cores: 128  ram: 478 GB  disk: 50382 GB  A30: 3
  rutg-w2.fabric-testbed.net  |  cores: 56  ram: 254 GB  disk: 49182 GB  A30: 1
  rutg-w4.fabric-testbed.net  |  cores: 108  ram: 318 GB  disk: 833 GB  A30: 1
  kans-w3.fabric-testbed.net  |  cores: 46  ram: 310 GB  disk: 533 GB  A30: 1
  amst-w1.fabric-testbed.net  |  cores: 70  ram: 294 GB  disk: 58102 GB  A30: 2
  amst-w3.fabric-testbed.net  |  cores: 80  ram: 382 GB  disk: 733 GB  A30: 1
  brist-w1.fabric-testbed.net  |  cores: 30  ram: 166 GB  disk: 47742 GB  A30: 1
  sri-w1.f

In [18]:
find_hosts_with_resources(
    fablib,
    min_cores=20,
    min_ram=64,
    gpu_rtx6000=1,
    force_refresh=True,
    return_data=False
)

🔎 Searching for hosts with:
   • Cores: >= 20
   • RAM (GB): >= 64
   • RTX6000: >= 1

✅ Found 7 matching host(s):

  star-w2.fabric-testbed.net  |  cores: 48  ram: 302 GB  disk: 48082 GB  RTX6000: 3
  salt-w1.fabric-testbed.net  |  cores: 46  ram: 238 GB  disk: 58995 GB  RTX6000: 2
  gpn-w2.fabric-testbed.net  |  cores: 32  ram: 286 GB  disk: 25154 GB  RTX6000: 3
  tacc-w1.fabric-testbed.net  |  cores: 56  ram: 382 GB  disk: 49482 GB  RTX6000: 3
  utah-w2.fabric-testbed.net  |  cores: 58  ram: 126 GB  disk: 45572 GB  RTX6000: 2
  utah-w1.fabric-testbed.net  |  cores: 38  ram: 262 GB  disk: 49072 GB  RTX6000: 1
  dall-w1.fabric-testbed.net  |  cores: 22  ram: 206 GB  disk: 57795 GB  RTX6000: 1


## 12. Find Best Site for Your Topology

In [19]:
import uuid
from typing import List

# ── Component model normalisation ─────────────────────────────────────────────
# Maps upper-cased topology model strings to the exact strings that
# fablib's add_component() expects.
_COMPONENT_MODEL_MAP = {
    # GPUs
    'GPU_RTX6000':        'GPU_RTX6000',
    'GPU_TESLAT4':        'GPU_TeslaT4',
    'GPU_TESLA_T4':       'GPU_TeslaT4',
    'GPU_A30':            'GPU_A30',
    'GPU_A40':            'GPU_A40',
    # NICs / DPUs
    'NIC_BASIC':          'NIC_Basic',
    'NIC_CONNECTX_5':     'NIC_ConnectX_5',
    'NIC_CONNECTX_6':     'NIC_ConnectX_6',
    'NIC_CONNECTX_7_100': 'NIC_ConnectX_7_100',
    'NIC_CONNECTX_7_400': 'NIC_ConnectX_7_400',
    # FPGAs
    'FPGA_XILINX_U280':   'FPGA_Xilinx_U280',
    # NVMe
    'NVME_P4510':         'NVME_P4510',
}

def _normalize_model(model_str):
    """Return the canonical fablib component model string."""
    return _COMPONENT_MODEL_MAP.get(
        model_str.upper().replace('-', '_'), model_str
    )


def _build_probe_slice(fablib, topology, sites_prefer):
    """
    Build a temporary (never submitted) slice from a topology model.
    Used exclusively as input to slice.validate().
    """
    probe = fablib.new_slice(name=f'_probe_{uuid.uuid4().hex[:8]}')
    for node in topology.site_topology_nodes.iter_nodes():
        site = node.site if node.site else (sites_prefer[0] if sites_prefer else None)
        fab_node = probe.add_node(
            name=node.hostname,
            site=site,
            cores=node.capacity.cpu,
            ram=node.capacity.ram,
            disk=node.capacity.disk,
        )
        for comp_name, gpu  in node.pci.gpu.items():
            fab_node.add_component(model=_normalize_model(gpu.model),  name=comp_name)
        for comp_name, fpga in node.pci.fpga.items():
            fab_node.add_component(model=_normalize_model(fpga.model), name=comp_name)
        for comp_name, nic  in node.pci.network.items():
            fab_node.add_component(model=_normalize_model(nic.model),  name=comp_name)
        for comp_name, dpu  in node.pci.dpu.items():
            fab_node.add_component(model=_normalize_model(dpu.model),  name=comp_name)
        for comp_name, nvme in node.pci.nvme.items():
            fab_node.add_component(model=_normalize_model(nvme.model), name=comp_name)
    return probe


def _node_host_requirements(node):
    """
    Extract host-level resource requirements from a topology node as a
    dict keyed by list_hosts() field names.
    """
    req = {
        'cores_available': node.capacity.cpu,
        'ram_available':   node.capacity.ram,
        'disk_available':  node.capacity.disk,
    }
    for _, gpu in node.pci.gpu.items():
        m = gpu.model.upper().replace('-', '_')
        if   'RTX6000'  in m or 'RTX_6000' in m: req['rtx6000_available']   = req.get('rtx6000_available',   0) + 1
        elif 'TESLAT4'  in m or 'TESLA_T4' in m:  req['tesla_t4_available'] = req.get('tesla_t4_available', 0) + 1
        elif 'A30'      in m:                      req['a30_available']      = req.get('a30_available',      0) + 1
        elif 'A40'      in m:                      req['a40_available']      = req.get('a40_available',      0) + 1
    for _, fpga in node.pci.fpga.items():
        if 'U280' in fpga.model.upper():
            req['fpga-xilinx-u280_available'] = req.get('fpga-xilinx-u280_available', 0) + 1
    for _, nvme in node.pci.nvme.items():
        req['nvme-p4510_available'] = req.get('nvme-p4510_available', 0) + 1
    for _, nic in node.pci.network.items():
        m = nic.model.upper().replace('-', '_')
        if   'CONNECTX_7_400' in m: req['smartnic-connectx-7-400_available'] = req.get('smartnic-connectx-7-400_available', 0) + 1
        elif 'CONNECTX_7_100' in m: req['smartnic-connectx-7-100_available'] = req.get('smartnic-connectx-7-100_available', 0) + 1
        elif 'CONNECTX_6'     in m:
            if 'SHARED' in m:       req['sharednic-connectx-6_available']    = req.get('sharednic-connectx-6_available',    0) + 1
            else:                   req['smartnic-connectx-6_available']      = req.get('smartnic-connectx-6_available',      0) + 1
        elif 'CONNECTX_5'     in m: req['smartnic-connectx-5_available']      = req.get('smartnic-connectx-5_available',      0) + 1
    for _, dpu in node.pci.dpu.items():
        m = dpu.model.upper().replace('-', '_')
        if   'CONNECTX_7_400' in m: req['smartnic-connectx-7-400_available'] = req.get('smartnic-connectx-7-400_available', 0) + 1
        elif 'CONNECTX_7_100' in m: req['smartnic-connectx-7-100_available'] = req.get('smartnic-connectx-7-100_available', 0) + 1
    return req


def _make_host_filter(node_site, req, sites_prefer, sites_avoid):
    """Return a list_hosts() filter function scoped to a single topology node."""
    def host_filter(row):
        host_site = row.get('name', '').split('-')[0].upper()
        if node_site and host_site != node_site:
            return False
        if not node_site and sites_prefer:
            if not any(s.upper() == host_site for s in sites_prefer):
                return False
        if sites_avoid:
            if any(s.upper() == host_site for s in sites_avoid):
                return False
        if row.get('state') != 'Active':
            return False
        for field, min_val in req.items():
            if min_val > 0 and row.get(field, 0) < min_val:
                return False
        return True
    return host_filter


# ─────────────────────────────────────────────────────────────────────────────

def validate_topology(
    fablib,
    topology,
    sites_prefer: List[str] = None,
    return_data:  bool = False,
):
    """
    Validate a topology against current FABRIC resources using the native
    slice.validate() API — the same check FABRIC runs before actual deployment.

    Builds a temporary probe slice (never submitted) from the topology model
    and calls validate() to get a per-node pass/fail result with FABRIC's
    own error messages.

    Args:
        fablib:       FablibManager instance
        topology:     SiteTopology model or dict
        sites_prefer: Preferred site names for nodes with no fixed site
        return_data:  Return result dict; False (default) suppresses
                      Jupyter auto-display

    Returns:
        dict if return_data=True, else None
        {
          'can_deploy':        bool,
          'validation_errors': {node_name: error_str},
        }
    """
    from fabric_generic_cluster import load_topology_from_dict

    if isinstance(topology, dict):
        topology = load_topology_from_dict(topology)

    sites_prefer   = sites_prefer or []
    topology_nodes = list(topology.site_topology_nodes.iter_nodes())

    print(f'📊 Topology: {len(topology_nodes)} node(s)\n')
    print('🔎 Validating with FABRIC resource manager...\n')

    try:
        probe = _build_probe_slice(fablib, topology, sites_prefer)
        is_valid, errors = probe.validate(raise_exception=False)
    except Exception as e:
        print(f'❌ Validation call failed: {e}')
        result = {'can_deploy': False, 'validation_errors': {}}
        return result if return_data else None

    # Per-node summary
    for node in topology_nodes:
        err      = errors.get(node.hostname)
        status   = '✅' if not err else '❌'
        site_lbl = f' @ {node.site}' if node.site else ''
        comps    = (
            [g.model for _, g in node.pci.gpu.items()]     +
            [f.model for _, f in node.pci.fpga.items()]    +
            [n.model for _, n in node.pci.network.items()] +
            [d.model for _, d in node.pci.dpu.items()]     +
            [v.model for _, v in node.pci.nvme.items()]
        )
        comp_str = f'  [{", ".join(comps)}]' if comps else ''
        print(f'  {status} {node.hostname}{site_lbl} — '
              f'{node.capacity.cpu} cores, {node.capacity.ram} GB RAM, '
              f'{node.capacity.disk} GB disk{comp_str}')
        if err:
            print(f'       ↳ {err}')
    print()

    print('=' * 70)
    if is_valid:
        print('✅ ✅ ✅  TOPOLOGY CAN BE DEPLOYED  ✅ ✅ ✅')
    else:
        failed = list(errors.keys())
        print('❌  TOPOLOGY CANNOT BE DEPLOYED')
        print(f'    {len(failed)} node(s) failed: {", ".join(failed)}')
    print('=' * 70)

    result = {'can_deploy': is_valid, 'validation_errors': errors}
    return result if return_data else None


In [20]:
def find_hosts_for_topology(
    fablib,
    topology,
    sites_prefer:  List[str] = None,
    sites_avoid:   List[str] = None,
    force_refresh: bool = False,
    return_data:   bool = False,
):
    """
    Find candidate worker hosts for each node in a topology.

    Calls fablib.list_hosts() per topology node and lists every worker host
    that individually satisfies the node's resource requirements.
    All candidates are shown — no random selection.

    Run validate_topology() first to confirm resources are available before
    calling this function.

    Args:
        fablib:        FablibManager instance
        topology:      SiteTopology model or dict
        sites_prefer:  Preferred site names for nodes with no fixed site
        sites_avoid:   Sites to exclude
        force_refresh: Bypass resource cache
        return_data:   Return result dict; False (default) suppresses
                       Jupyter auto-display

    Returns:
        dict if return_data=True, else None
        {
          'matches': [{'topology_node', 'site', 'candidate_hosts',
                       'requirements_met'}, ...]
        }
    """
    from fabric_generic_cluster import load_topology_from_dict

    if isinstance(topology, dict):
        topology = load_topology_from_dict(topology)

    sites_prefer   = sites_prefer or []
    sites_avoid    = sites_avoid  or []
    topology_nodes = list(topology.site_topology_nodes.iter_nodes())

    print(f'📊 Topology: {len(topology_nodes)} node(s)\n')
    print('🔍 Finding candidate worker hosts per node...\n')

    host_fields = [
        'name', 'state',
        'cores_available', 'ram_available', 'disk_available',
        'rtx6000_available', 'tesla_t4_available', 'a30_available', 'a40_available',
        'nvme-p4510_available',
        'smartnic-connectx-5_available', 'smartnic-connectx-6_available',
        'smartnic-connectx-7-100_available', 'smartnic-connectx-7-400_available',
        'sharednic-connectx-6_available',
        'fpga-xilinx-u280_available',
    ]

    all_matches = []

    for idx, node in enumerate(topology_nodes, 1):
        node_site = node.site.upper() if node.site else None
        req       = _node_host_requirements(node)

        print(f'  Node {idx}/{len(topology_nodes)}: {node.hostname}')

        try:
            candidates = fablib.list_hosts(
                fields=host_fields,
                pretty_names=False,
                filter_function=_make_host_filter(node_site, req, sites_prefer, sites_avoid),
                output='list',
                quiet=True,
                force_refresh=force_refresh,
            )
        except Exception as e:
            print(f'    ❌ list_hosts error: {e}')
            candidates = []

        if candidates:
            host_site = candidates[0]['name'].split('-')[0].upper()
            print(f'    ✅ {len(candidates)} candidate host(s) at {host_site}:')
            for h in candidates:
                line = (f'       • {h["name"]}'
                        f'  |  cores: {h.get("cores_available")}'
                        f'  ram: {h.get("ram_available")} GB'
                        f'  disk: {h.get("disk_available")} GB')
                for _param, (label, _sf, hf) in _FIELD_LOOKUP.items():
                    if hf not in ('cores_available', 'ram_available', 'disk_available'):
                        if h.get(hf, 0) > 0:
                            line += f'  {label}: {h[hf]}'
                print(line)
            all_matches.append({
                'topology_node':   node.hostname,
                'site':            host_site,
                'candidate_hosts': [h['name'] for h in candidates],
                'requirements_met': True,
            })
        else:
            assigned_site = node_site or (sites_prefer[0].upper() if sites_prefer else 'any')
            print(f'    ⚠️  No specific worker found for {node.hostname}')
            all_matches.append({
                'topology_node':   node.hostname,
                'site':            assigned_site,
                'candidate_hosts': [],
                'requirements_met': False,
            })
        print()

    # Summary
    successful = [m for m in all_matches if m['requirements_met']]
    failed     = [m for m in all_matches if not m['requirements_met']]

    print('=' * 70)
    if failed:
        print(f'⚠️  {len(failed)} node(s) had no matching host: '
              f'{", ".join(m["topology_node"] for m in failed)}')
    else:
        sites_used = {}
        for m in all_matches:
            sites_used[m['site']] = sites_used.get(m['site'], 0) + 1
        print('📍 Sites:')
        for site, count in sites_used.items():
            print(f'   {site}: {count} node(s)')
        print()
        print('📋 Candidate hosts per node:')
        for m in all_matches:
            hosts = m['candidate_hosts']
            shown = ', '.join(hosts[:3])
            if len(hosts) > 3:
                shown += f' (+{len(hosts) - 3} more)'
            print(f'   {m["topology_node"]} → {shown}')
    print('=' * 70)

    result = {'matches': all_matches}
    return result if return_data else None


### Load Topology from the Model

In [32]:
import sys
from pathlib import Path

# Add parent directory to Python path to import modules
repo_root = Path.cwd().parent

print(f"✅ Python path configured")
print(f"   Repository root: {repo_root}")

✅ Python path configured
   Repository root: /home/fabric/work/fabric-generic-cluster-notebooks


<div class="alert alert-block alert-info">
Define the <b>YAML Directory</b> and <b>Model File</b>
</div>

In [33]:
YAML_DIR = repo_root / "model"
site_topology_yaml = "../model/m2-2.yml"

print(f"✅ YAML directory: {YAML_DIR}")
print(f"✅ YAML file: {site_topology_yaml}")

✅ YAML directory: /home/fabric/work/fabric-generic-cluster-notebooks/model
✅ YAML file: ../model/m2-2.yml


In [34]:
import sys
from pathlib import Path

# Add parent directory to Python path to import modules
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

print(f"✅ Python path configured")
print(f"   Repository root: {repo_root}")

# Import modules

from fabric_generic_cluster import load_topology_from_yaml_file, SiteTopology
from fabric_generic_cluster import deployment as sd
from fabric_generic_cluster import network_config as snc
from fabric_generic_cluster import ssh_setup as ssh

print("✅ Modules imported successfully")


# Load and validate topology (raises ValidationError if invalid)
try:
    topology = load_topology_from_yaml_file(site_topology_yaml)
    print("✅ Topology loaded and validated successfully!")
    print(f"   Nodes: {len(topology.site_topology_nodes.nodes)}")
    print(f"   Networks: {len(topology.site_topology_networks.networks)}")

    result = validate_topology(
        fablib,
        topology,
        sites_prefer=[],  # Optional: prefer certain sites ['WASH', 'SRI']
        return_data=True,
    )
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise


# Access results
if result and result['can_deploy']:
    print("\n✅ Topology validated — run find_hosts_for_topology() to see candidate hosts.")



✅ Python path configured
   Repository root: /home/fabric/work/fabric-generic-cluster-notebooks
✅ Modules imported successfully
✅ Topology loaded and validated successfully!
   Nodes: 2
   Networks: 3
📊 Topology: 2 node(s)

🔎 Validating with FABRIC resource manager...

  ❌ lc-1 @ MASS — 16 cores, 64 GB RAM, 100 GB disk  [GPU_RTX6000, NIC_Basic, NIC_Basic]
       ↳ Invalid Request: Requested Node cannot be accommodated by any of the hosts on site: MASS. Details: Invalid Request: Host: mass-w2.fabric-testbed.net does not have the requested component: GPU-RTX6000.
  ✅ lc-2 @ GPN — 16 cores, 64 GB RAM, 100 GB disk  [GPU_RTX6000, NIC_Basic, NIC_Basic]

❌  TOPOLOGY CANNOT BE DEPLOYED
    1 node(s) failed: lc-1


#### Example: Validate topology against FABRIC resources

In [35]:
#### Example usage
validate_topology(
    fablib,
    topology,
    sites_prefer=['WASH', 'SRI'],
)

📊 Topology: 2 node(s)

🔎 Validating with FABRIC resource manager...

  ❌ lc-1 @ MASS — 16 cores, 64 GB RAM, 100 GB disk  [GPU_RTX6000, NIC_Basic, NIC_Basic]
       ↳ Invalid Request: Requested Node cannot be accommodated by any of the hosts on site: MASS. Details: Invalid Request: Host: mass-w2.fabric-testbed.net does not have the requested component: GPU-RTX6000.
  ✅ lc-2 @ GPN — 16 cores, 64 GB RAM, 100 GB disk  [GPU_RTX6000, NIC_Basic, NIC_Basic]

❌  TOPOLOGY CANNOT BE DEPLOYED
    1 node(s) failed: lc-1


#### Example: Find candidate hosts per node

In [ ]:
find_hosts_for_topology(
    fablib,
    topology,
    sites_prefer=['WASH', 'SRI'],
    sites_avoid=[],
)

## 13. Export Site Availability to File

In [28]:
def export_site_availability(fablib, filename="site_availability.txt"):
    """
    Export current site availability to file.
    
    Args:
        fablib: FablibManager instance
        filename: Output filename
    """
    from datetime import datetime
    
    # Get all sites
    sites_df = get_sites_dataframe(fablib)
    
    # Add timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(filename, 'w') as f:
        f.write(f"FABRIC Site Availability Report\n")
        f.write(f"Generated: {timestamp}\n")
        f.write("=" * 80 + "\n\n")
        f.write(sites_df.to_string())
    
    print(f"✅ Exported to: {filename}")

In [29]:
# Uncomment to export
export_site_availability(fablib)

✅ Exported to: site_availability.txt


## 14. Get All Available Resource Types

In [30]:
# Get all sites as DataFrame
sites_df = get_sites_dataframe(fablib)

# Find columns related to hardware
hardware_columns = [
    col for col in sites_df.columns 
    if any(hw in col.lower() for hw in [
        'nic', 'gpu', 'fpga', 'nvme', 'cores', 'ram', 'disk', 
        'connectx', 'tesla', 'rtx', 'a30', 'a40',
        'u280', 'smartnic', 'sharednic'
    ]) and 'available' in col.lower()
]

print("📋 Available Resource Types in FABRIC:\n")
for col in sorted(hardware_columns):
    if col in sites_df.columns:
        # Handle NaN values properly
        total_available = sites_df[col].fillna(0).sum() if pd.api.types.is_numeric_dtype(sites_df[col]) else "N/A"
        if isinstance(total_available, float):
            total_available = int(total_available)
        print(f"   • {col}: {total_available}")


📋 Available Resource Types in FABRIC:

   • cores_available: 79721
   • disk_available: 1848033
   • fpga-xilinx-u280_available: 18
   • gpu-a30_available: 27
   • gpu-a40_available: 0
   • gpu-rtx6000_available: 20
   • gpu-tesla t4_available: 16
   • nvme-p4510_available: 347
   • ram_available: 39066
   • sharednic-connectx-6_available: 16400
   • smartnic-connectx-5_available: 75
   • smartnic-connectx-6_available: 45
   • smartnic-connectx-7-100_available: 5
   • smartnic-connectx-7-400_available: 2


## 15. Summary and Use Cases

In [31]:
print("""
═══════════════════════════════════════════════════════════════════════════════
                    FABRIC Resource Finder - Summary
═══════════════════════════════════════════════════════════════════════════════

This notebook provides comprehensive tools for finding FABRIC resources:

📌 Quick Queries:
   • Find sites with specific NICs (ConnectX-5, ConnectX-6, ConnectX-7)
   • Find sites with GPUs (RTX6000, A30, A40, Tesla T4)
   • Find sites with FPGAs (Xilinx U280)
   • Find sites with NVMe storage

📌 Advanced Features:
   • Custom multi-criteria resource finder
   • Automatic site selection for topology
   • Export results to file
   • List all available resource types

📌 Use Cases:
   • Planning: Find suitable sites before deployment
   • Optimization: Choose best site for your workload
   • Monitoring: Track resource availability over time

📌 Key Functions:
   • get_sites_dataframe(fablib) - Get all site data as DataFrame
   • find_sites_with_resources(...) - Find sites matching criteria
   • find_best_site_for_topology(fablib, topology) - Match topology to site
   • export_site_availability(fablib, filename) - Export to file

═══════════════════════════════════════════════════════════════════════════════
""")


═══════════════════════════════════════════════════════════════════════════════
                    FABRIC Resource Finder - Summary
═══════════════════════════════════════════════════════════════════════════════

This notebook provides comprehensive tools for finding FABRIC resources:

📌 Quick Queries:
   • Find sites with specific NICs (ConnectX-5, ConnectX-6, ConnectX-7)
   • Find sites with GPUs (RTX6000, A30, A40, Tesla T4)
   • Find sites with FPGAs (Xilinx U280)
   • Find sites with NVMe storage

📌 Advanced Features:
   • Custom multi-criteria resource finder
   • Automatic site selection for topology
   • Export results to file
   • List all available resource types

📌 Use Cases:
   • Planning: Find suitable sites before deployment
   • Optimization: Choose best site for your workload
   • Monitoring: Track resource availability over time

📌 Key Functions:
   • get_sites_dataframe(fablib) - Get all site data as DataFrame
   • find_sites_with_resources(...) - Find sites matchin